In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_2007_2015_bn.csv")

In [3]:
df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_lv

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,PR,RQ,COC,ROL,CPI,EUPC
0,9.22,3.11,5.39,10.18,35.0,-0.82,22.75,43.665876,18.600884,11.319401,...,70.435714,0.02,6.80,424.0,50,0.95,0.34,0.72,48.0,4.468538e+07
1,4.20,2.85,5.59,7.96,35.0,-1.05,22.84,36.975656,20.701069,-2.368037,...,70.435714,0.01,7.81,-338.0,55,0.97,0.23,0.78,50.0,6.940880e+07
2,-0.58,3.19,5.48,8.75,34.0,-1.65,21.52,23.211232,19.573897,-14.642313,...,66.200000,-0.05,-3.95,-780.0,55,0.92,0.20,0.79,45.0,1.985811e+08
3,2.02,3.11,5.44,8.61,34.0,-2.08,20.86,20.931345,18.789970,-1.635144,...,64.200000,-0.04,16.38,1140.0,55,0.93,0.20,0.76,43.0,2.209354e+08
4,5.72,3.10,5.17,8.90,33.0,-1.82,21.42,26.774564,19.255704,4.929580,...,65.100000,-0.03,75.29,-641.0,50,0.95,0.28,0.74,42.0,2.485579e+08
5,3.99,3.16,5.27,8.37,33.0,-1.24,21.42,28.639673,17.988056,8.644500,...,66.200000,0.01,212.38,-201.0,50,1.01,0.24,0.78,49.0,3.518108e+08
6,3.39,3.00,5.39,8.06,32.0,-1.07,20.82,25.298238,18.281111,3.195452,...,67.500000,-0.02,218.90,-72.0,50,1.04,0.32,0.76,53.0,3.102758e+08
7,3.45,3.09,5.35,7.50,32.0,-0.94,19.62,24.665948,18.488335,3.060974,...,68.400000,-0.01,187.31,-55.0,50,1.17,0.41,0.87,55.0,3.581094e+08
8,3.09,3.31,5.37,7.94,32.0,-0.82,19.78,24.149737,18.709564,4.636452,...,70.500000,0.00,107.29,44.0,50,1.07,0.43,0.77,55.0,3.044853e+08


In [5]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['EUPC_3'] = lv['EUPC'].shift(3)
lv['EBSG_1'] = lv['EBSG'].shift(1)
lv['EIAP_3'] = lv['EIAP'].shift(3)
lv['FDI_1'] = lv['FDI'].shift(1)
lv['RQ_3'] = lv['RQ'].shift(3)
lv['VABI_1'] = lv['VABI'].shift(1)
lv['FDI_3'] = lv['FDI'].shift(3)
lv['GGAG_3'] = lv['GGAG'].shift(3)

In [6]:
lv = lv.dropna()

In [7]:
sm = StructureModel()

In [8]:
sm.add_edges_from([
    ('EUPC_3', 'FDI'),
])

In [9]:
sm.edges

OutEdgeView([('EUPC_3', 'FDI')])

In [10]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lt_2016_2022.html")

Graphs/fully_connected_lt_2016_2022.html


In [11]:
bn = BayesianNetwork(sm)

In [12]:
discretised_lv = pd.DataFrame(index=lv.index)

for col in lv.columns:
    no_unique = lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lv)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  CPI  EUPC  \
3    0    1    2     2     1    0     1     0     2      0  ...    0     0   
4    2    1    0     2     0    0     2     2     2      2  ...    0     0   
5    2    2    0     1     0    1     2     2     0      2  ...    1     2   
6    1    0    2     1     0    1     1     1     0      1  ...    1     1   
7    1    0    1     0     0    2     0     1     1      0  ...    2     2   
8    0    2    1     0     0    2     0     0     1      1  ...    2     1   

   EUPC_3  EBSG_1  EIAP_3  FDI_1  RQ_3  VABI_1  FDI_3  GGAG_3  
3       0       2       2      0     1       2      2       2  
4       0       2       0      0     2       1      1       1  
5       1       0       1      2     0       1      0       0  
6       1       0       1      2     0       1      0       2  
7       2       1       2      1     1       0      2       0  
8       2       1       0      1     2       0     

In [13]:
discretised_lv = discretised_lv.reset_index(drop=True)
bn.fit_node_states(discretised_lv)
baseline_auc = utils.get_avg_auc_all_info(discretised_lv, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.07489275932312 seconds
Processing fold 1 using 7 cores takes 3.4641520977020264 seconds
Processing fold 2 using 7 cores takes 3.0853569507598877 seconds
Processing fold 3 using 7 cores takes 2.932321071624756 seconds
Processing fold 4 using 7 cores takes 2.9513421058654785 seconds
Baseline AUC: 0.15


In [14]:
edges_to_add = [('LV', 'EUPC_3'), ('LV', 'FDI')]
edges_to_remove = [('EUPC_3', 'FDI')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [15]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_2016_2022.html")

Graphs/node_added_lt_2016_2022.html


In [16]:
discretised_lv['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lv, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_3' and 'FDI': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.1352477073669434 seconds
Processing fold 1 using 7 cores takes 3.0973517894744873 seconds
Processing fold 2 using 7 cores takes 3.033527135848999 seconds
Processing fold 3 using 7 cores takes 3.0279319286346436 seconds
Processing fold 4 using 7 cores takes 3.124755859375 seconds
AUC from adding LV between 'EUPC_3' and 'FDI': 0.05
